# 05 — Rollout evaluation (dynamic_dt)

3-step autoregressive rollout on **2023 test** data.

| Knob | Description |
|------|-------------|
| `RUN_ALL_DT` | **`True`** (default) → evaluate all 8 dt in parallel on **cuda:0–7** |
| `DT_DAYS` | Single dt when `RUN_ALL_DT=False` |
| `EXPORT_KEY` | Must match training window (uv, iD=2) |

**Sequence:** 5 cond frames (`t-4…t`) → predict `t+1`. **Test metric:** speed RMSE.

**dt → GPU:** 3→0, 5→1, 7→2, 10→3, 12→4, 15→5, 17→6, 20→7

Horizons: **+dt**, **+2dt**, **+3dt**. Sample plots → `Test&Rollout/.../samples/sample_{i}.png`

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MW_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "dynamic_dt" else (
    NOTEBOOK_DIR if NOTEBOOK_DIR.name == "Magic_window" else NOTEBOOK_DIR / "Magic_window"
)
DT_DIR = MW_DIR / "dynamic_dt"
ROOT = MW_DIR.parent
for _p in (ROOT, MW_DIR):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
import dynamic_dt.multi_dt_runner as _mdr
import dynamic_dt.rollout_runner as _rr
importlib.reload(_mdr)
importlib.reload(_rr)
from dynamic_dt.device_plan import format_dt_device_plan
from dynamic_dt.dt_config import dt_hours, forecast_horizons_days
from dynamic_dt.window_catalog import get_window_spec, primary_export_component
from dynamic_dt.window_paths import test_prefix
from dynamic_dt.run_registry import resolve_model_checkpoint, run_dir, run_name, weight_decay_run_tag, RUN_TAG
from dynamic_dt.rollout_runner import rollout_one_dt
from dynamic_dt.multi_dt_runner import launch_rollout_all, print_job_summary

LOG_DIR = DT_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ============ USER KNOBS ============
EXPORT_KEY = "magicwin_reference"   # or "smallfixwin"
RUN_ALL_DT = True                   # all 8 dt on GPUs 0–7
DT_DAYS = 15                        # only when RUN_ALL_DT=False
WEIGHT_DECAY = 1e-3
MODEL_RUN_DIR = None
PLOT_INDICES = [0, 1, 2, 32, 45]
DETACHED = True
# ====================================

WINDOW = get_window_spec(EXPORT_KEY)
print(f"EXPORT_KEY={EXPORT_KEY} ({WINDOW['label']}) | RUN_ALL_DT={RUN_ALL_DT}")
print(format_dt_device_plan(None if RUN_ALL_DT else [DT_DAYS]))

In [ ]:
if RUN_ALL_DT:
    jobs = launch_rollout_all(
        dt_dir=DT_DIR,
        root=ROOT,
        export_key=EXPORT_KEY,
        log_dir=LOG_DIR,
        plot_indices=PLOT_INDICES,
        detached=DETACHED,
        weight_decay=WEIGHT_DECAY,
    )
    print_job_summary(jobs, label="Rollout")
else:
    from dynamic_dt.rollout_core import evaluate_rollout
    from dynamic_dt.run_registry import save_rollout_results, test_rollout_dir, update_comparison_index

    export_component = primary_export_component(EXPORT_KEY)
    channel_names = WINDOW["channel_names_rollout"]
    dt = dt_hours(DT_DAYS)
    data_prefix = test_prefix(DT_DIR, EXPORT_KEY, component=export_component, dt_hours=dt)
    models_dir = DT_DIR / "Models" / "adv_no"
    run_tag = weight_decay_run_tag(RUN_TAG, WEIGHT_DECAY)
    run_name_value = run_name(EXPORT_KEY, dt_hours=dt, run_tag=run_tag)
    run_dir_path = Path(MODEL_RUN_DIR) if MODEL_RUN_DIR else run_dir(
        models_dir, EXPORT_KEY, dt_hours=dt, run_tag=run_tag
    )
    test_rollout_path = test_rollout_dir(DT_DIR, EXPORT_KEY, dt_hours=dt, run_tag=run_tag)
    train_cfg, model_path, stats_path, _, _, _ = resolve_model_checkpoint(
        run_dir_path, expected_run_name=run_name_value
    )
    h1, h2, h3 = forecast_horizons_days(DT_DAYS)
    print(f"DT_DAYS={DT_DAYS} horizons +{h1}/+{h2}/+{h3}d | model={model_path}")

    if DETACHED:
        launched = rollout_one_dt(
            dt_dir=DT_DIR, root=ROOT, export_key=EXPORT_KEY, dt_days=DT_DAYS,
            log_dir=LOG_DIR, plot_indices=PLOT_INDICES, detached=True,
            weight_decay=WEIGHT_DECAY, model_run_dir=MODEL_RUN_DIR,
        )
        print(f"Started PID {launched['pid']} | tail -f {launched['log_path']}")
    else:
        from dynamic_dt.device_plan import cuda_visible_devices
        import os
        os.environ["CUDA_VISIBLE_DEVICES"] = cuda_visible_devices(DT_DAYS)
        results = evaluate_rollout(
            data_prefix=data_prefix,
            model_path=model_path,
            stats_path=stats_path,
            dt_days=DT_DAYS,
            train_cfg=train_cfg,
            output_dir=test_rollout_path,
            device="cuda:0",
            plot_indices=PLOT_INDICES,
            channel_names=channel_names,
        )
        save_rollout_results(test_rollout_path, results)
        update_comparison_index(
            models_dir, run_name_value=run_name_value, export_key=EXPORT_KEY,
            run_dir_path=run_dir_path, rollout_results=results,
        )
        print(f"Saved {test_rollout_path / 'rollout_results.json'}")